# Lakehouse-Pattern — Bronze → Silver → Gold walkthrough

This notebook walks the medallion pipeline end-to-end using the same
Python modules that `make dag` invokes. It exists to give reviewers
(and future-you) a scrollable narrative view of what each layer looks
like, without needing to run Streamlit.

**Prerequisites.** Run `make preflight` first to confirm Java 17+ is
available. This notebook assumes you launched Jupyter from the repo
root (`jupyter lab`) so the `lakehouse` package is importable.

**Runtime.** ~2–3 minutes on a laptop for the ETL stages. The optional
final section that trains an MLflow model adds ~15 seconds.


In [ ]:
# Notebook boilerplate — make imports resolve from the repo root.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


## 1. Preflight: Java 17+ check

The `lakehouse.env` module runs an explicit Java version check before
we build a `SparkSession`. Delta Lake 3.2 requires Java 17 or newer;
older JVMs fail with cryptic serialization errors that are easy to
misdiagnose. The preflight surfaces a friendly, actionable error early.


In [ ]:
from lakehouse import env

env.check_prerequisites()  # raises PrerequisiteError with a fix hint if Java is missing/old


## 2. Generate the sample dataset

We ship a deterministic synthetic loader so this notebook — and CI —
never depend on external network access. Real production ingestion
would replace this cell with a read from S3 / GCS / a broker.


In [ ]:
from data.sample_raw.download import main as download_sample

download_sample()


## 3. Bronze — append-only raw

Bronze is the replayable source of truth. Rows land exactly as they
arrived: no type coercion, no dedup, no validation. The only Delta
feature we lean on here is **schema enforcement** — writes with
unexpected columns are rejected (see `tests/test_delta_features.py`).


In [ ]:
from ingestion.batch_ingest import run as ingest_bronze

ingest_bronze()


In [ ]:
from lakehouse import paths
from lakehouse.spark import get_spark

spark = get_spark("notebook")
bronze = spark.read.format("delta").load(str(paths.BRONZE_TRANSACTIONS))
print(f"Bronze row count: {bronze.count():,}")
bronze.show(5, truncate=False)


## 4. Silver — typed, deduped, quality-gated

Silver is where we earn our keep. We MERGE upsert on `transaction_id`
to make ingestion idempotent, partition by `event_date` (low-cardinality,
always in filter predicates), Z-ORDER by `customer_id` (high-cardinality,
frequently filtered), and quarantine any expectation failures into
`silver.transactions_rejects` rather than dropping them silently.

`OPTIMIZE` + `VACUUM` run in-pipeline for demo simplicity; in production
they'd move to a scheduled maintenance job so they don't lengthen the
critical path.


In [ ]:
from transform.silver_clean import run as build_silver

build_silver()


In [ ]:
silver = spark.read.format("delta").load(str(paths.SILVER_TRANSACTIONS))
print(f"Silver row count: {silver.count():,}")
silver.printSchema()
silver.show(5, truncate=False)


## 5. Gold — business marts

Two Gold tables:

- **`gold.daily_revenue`** — grain `(event_date, country)`. Powers the
  time-series chart in the Streamlit app.
- **`gold.customer_ltv`** — grain `customer_id`. Powers the top-N
  customer view and feeds the RAG index.


In [ ]:
from transform.gold_aggregate import run as build_gold

build_gold()


In [ ]:
daily = spark.read.format("delta").load(str(paths.GOLD_DAILY_REVENUE))
print("gold.daily_revenue:")
daily.orderBy("event_date", "country").show(10, truncate=False)

print("gold.customer_ltv (top 10 by lifetime revenue):")
ltv = spark.read.format("delta").load(str(paths.GOLD_CUSTOMER_LTV))
ltv.orderBy("lifetime_revenue", ascending=False).show(10, truncate=False)


## 6. Delta time travel

Every Delta write produces a new version in the `_delta_log`. Reading
`versionAsOf=0` gives us the very first snapshot of the Silver table —
useful for audits, reproducibility, and rollback.


In [ ]:
silver_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(str(paths.SILVER_TRANSACTIONS))
)
print(f"Silver @ v0 rows: {silver_v0.count():,}")
print(f"Silver @ HEAD rows: {silver.count():,}")


## 7. (Optional) MLflow training

Uncomment the cell below to train a small daily-revenue forecaster and
log the run to `mlruns/`. Skip this if you just wanted the ETL walkthrough
— nothing downstream depends on it.

After running, launch `mlflow ui --backend-store-uri mlruns` from the
repo root to inspect the run.


In [ ]:
# from ml.train_model import train
# train(num_boost_round=200, max_depth=8)


## What's next

- Launch the Streamlit gold explorer: `make serve` (opens
  http://localhost:8501).
- Rebuild the RAG index and ask a question:
  `python -m ml.rag_demo.rag_pipeline --rebuild --question "Who spent the most?"`
- Read the [architecture deep-dive](../docs/architecture.md) and the
  [concept-coverage matrix](../docs/concept-coverage.md) for the full tour.
